# 00 ML Regression Setup and Baselines

Lock down the dataset, target, and OLS / HAC-OLS baselines. Every later notebook in this section will compare against the numbers you record here.

## Table of Contents
- [Why this notebook matters](#why)
- [Environment bootstrap](#bootstrap)
- [Load data and define the prediction target](#load-data)
- [Time-aware train/test split](#split)
- [Baseline 1: OLS](#baseline-ols)
- [Baseline 2: OLS with HAC standard errors](#baseline-hac)
- [Walk-forward cross-validation](#walk-forward)
- [Record baselines](#record)
- [Reflection](#reflection)

<a id="why"></a>
## Why This Notebook Matters

Random Forest and XGBoost are flashy, but they are only useful if they beat a serious linear baseline on the *same* data with the *same* split. This notebook builds that baseline rigorously:

1. Pick a single target — next-quarter real GDP growth — and a fixed feature set.
2. Define one canonical time-aware train/test split.
3. Fit OLS and HAC-OLS, and record RMSE / MAE / R² on the test set.
4. Demonstrate `walk_forward_splits` from `src/evaluation.py` so you have a robust evaluator for the ML notebooks.

Open this notebook side-by-side with `02_regression/03_multifactor_regression_macro.ipynb`. The same target, same features, same data — different evaluation lens (out-of-sample RMSE rather than in-sample inference).

<a id="bootstrap"></a>
## Environment Bootstrap

In [ ]:
from __future__ import annotations

from pathlib import Path
import sys


def find_repo_root(start: Path) -> Path:
    p = start
    for _ in range(8):
        if (p / 'src').exists() and (p / 'docs').exists():
            return p
        p = p.parent
    raise RuntimeError('Could not find repo root.')


PROJECT_ROOT = find_repo_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
SAMPLE_DIR = PROJECT_ROOT / 'data' / 'sample'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
print('Repo root:', PROJECT_ROOT)

<a id="load-data"></a>
## Load Data and Define the Prediction Target

**Target.** Real GDP growth, quarter-over-quarter (`gdp_growth_qoq`). Predicting growth from *current-quarter* macro indicators is what most macro forecasters care about, and it gives a cleanly defined regression problem.

**Features.** Lagged macro indicators that are publicly available the moment GDP growth is being predicted: unemployment, fed funds rate, industrial production, retail sales, the 10y–2y treasury spread (lagged one quarter).

In [ ]:
import numpy as np
import pandas as pd

path = PROCESSED_DIR / 'macro_quarterly.csv'
if path.exists():
    df = pd.read_csv(path, index_col=0, parse_dates=True)
else:
    df = pd.read_csv(SAMPLE_DIR / 'macro_quarterly_sample.csv', index_col=0, parse_dates=True)

y_col = 'gdp_growth_qoq'
x_cols = [
    'T10Y2Y_lag1',
    'UNRATE_lag1',
    'FEDFUNDS_lag1',
    'INDPRO_lag1',
    'RSAFS_lag1',
]

data = df[[y_col] + x_cols].dropna().copy()
print('Shape:', data.shape)
print('Date range:', data.index.min(), '→', data.index.max())
data.head()

<a id="split"></a>
## Time-Aware Train/Test Split

Standard random splits **leak** information when applied to time series — the model sees the future. We use a chronological split: the first 80% of quarters train, the last 20% test. The helper `time_train_test_split_index` in `src/evaluation.py` does exactly this.

In [ ]:
from src.evaluation import time_train_test_split_index

split = time_train_test_split_index(len(data), test_size=0.2)
train = data.iloc[split.train_slice]
test = data.iloc[split.test_slice]

X_train = train[x_cols].to_numpy()
y_train = train[y_col].to_numpy()
X_test = test[x_cols].to_numpy()
y_test = test[y_col].to_numpy()

print(f'Train: {len(train)} quarters  ({train.index.min().date()} → {train.index.max().date()})')
print(f'Test:  {len(test)} quarters  ({test.index.min().date()} → {test.index.max().date()})')

<a id="baseline-ols"></a>
## Baseline 1: OLS

Vanilla ordinary least squares using `statsmodels`. We are not yet using HAC standard errors — that comes next. The point of this cell is to record the **out-of-sample** RMSE / MAE / R² that the ML models must beat.

In [ ]:
from src.econometrics import fit_ols
from src.evaluation import regression_metrics
import statsmodels.api as sm

ols = fit_ols(train, y_col=y_col, x_cols=x_cols)
print(ols.summary())

X_test_design = sm.add_constant(test[x_cols], has_constant='add')
yhat_test = ols.predict(X_test_design).to_numpy()
ols_metrics = regression_metrics(y_test, yhat_test)
ols_metrics

<a id="baseline-hac"></a>
## Baseline 2: OLS with HAC Standard Errors

HAC (Newey–West) does not change the **point estimates** — only the standard errors and the resulting t- and p-values. So the predictive RMSE / MAE / R² are identical to plain OLS. We still record HAC here because it is the right inference for time-series regressors and you will compare its t-stats against the ML models' (lack of) inference in the next two notebooks.

In [ ]:
from src.econometrics import fit_ols_hac

ols_hac = fit_ols_hac(train, y_col=y_col, x_cols=x_cols, maxlags=4)
print(ols_hac.summary())

yhat_test_hac = ols_hac.predict(X_test_design).to_numpy()
hac_metrics = regression_metrics(y_test, yhat_test_hac)
assert np.allclose(yhat_test, yhat_test_hac)  # confirms point estimates unchanged
hac_metrics

<a id="walk-forward"></a>
## Walk-Forward Cross-Validation

A single 80/20 split tells you how the model did on one tail of history. **Walk-forward CV** rolls the training window forward, refits, and scores the next chunk — repeatedly. The result is a distribution of RMSE values across many test windows, which is much more informative than a single number.

We will use this evaluator throughout the next two notebooks for hyperparameter tuning.

In [ ]:
from sklearn.linear_model import LinearRegression
from src.evaluation import walk_forward_splits

X_full = data[x_cols].to_numpy()
y_full = data[y_col].to_numpy()
n = len(data)

fold_rmse, fold_mae, fold_r2 = [], [], []
for sp in walk_forward_splits(n, initial_train_size=int(0.5 * n), test_size=8, step_size=8):
    Xtr, ytr = X_full[sp.train_slice], y_full[sp.train_slice]
    Xte, yte = X_full[sp.test_slice], y_full[sp.test_slice]
    m = LinearRegression().fit(Xtr, ytr)
    yp = m.predict(Xte)
    fold = regression_metrics(yte, yp)
    fold_rmse.append(fold['rmse']); fold_mae.append(fold['mae']); fold_r2.append(fold['r2'])

print(f'OLS walk-forward over {len(fold_rmse)} folds:')
print(f'  RMSE  mean={np.mean(fold_rmse):.4f}  median={np.median(fold_rmse):.4f}')
print(f'  MAE   mean={np.mean(fold_mae):.4f}')
print(f'  R²    mean={np.mean(fold_r2):.4f}  (negative R² means worse than predicting the mean)')

<a id="record"></a>
## Record Baselines

Write down these numbers. Notebooks `01` (Random Forest) and `02` (XGBoost) will compare against them.

| Model | RMSE (test) | MAE (test) | R² (test) |
|---|---|---|---|
| OLS (single 80/20 split) | (record from above) | | |
| OLS (walk-forward mean)  | (record from above) | | |

If the ML models cannot beat OLS on this exact dataset, that is a meaningful negative result — and a perfectly common one in macro forecasting, where small samples and persistent linear relationships favor simple models.

<a id="reflection"></a>
## Reflection

1. The HAC summary printed t-stats and p-values; the upcoming sklearn / xgboost models will not. Why is statistical inference harder for tree ensembles?
2. Walk-forward fold R² values can be negative on some folds. What does a negative R² mean, and which historical periods in this dataset are the most likely culprits?
3. If you knew that GDP growth was *highly nonlinear* in the term spread (e.g., a flat curve below 0% has a much stronger recession signal than below 1%), which baseline — OLS or a tree model — would you expect to win, and why?